[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/09-system-design/09_feeds_chat.ipynb)

*System Design · Notebook 9 of 10*

# News Feed, Chat & Notifications

Real-time social systems: fan-out on write vs read, the celebrity problem, merging and ranking timelines, author diversity, chat sequence numbers, unread counts, receipts, presence, routing to connection servers, group fan-out, reconnect sync, typing throttles, notification digests, quiet hours, channels, device tokens and sharded counters.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** Push fan-out is delivering a newspaper to every subscriber's door; pull is everyone walking to the newsstand. Celebrities need the newsstand.

**What interviewers really test here:** Fan-out on write vs read, hybrid feeds, ranking, chat ordering and sync, presence, routing to connection servers, notifications.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Design a news feed *(Basics)*
- [ ] 2. Design a chat system *(Core)*
- [ ] 3. Design a notification service *(Core)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import math
import json
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Push vs pull vs hybrid

|   | Push (write) | Pull (read) | Hybrid |
| --- | --- | --- | --- |
| Read cost | one lookup | merge many lists | lookup + few merges |
| Write cost | followers × posts | one insert | bounded |
| Celebrity | breaks | fine | pulled |
| Inactive users | wasted work | free | push only to active |

---
# Part 2: Lessons, step by step

## Lesson 1: Design a news feed

*Level: Basics*

**Requirements**: post, follow, see a feed of followed accounts (ranked or by time), very read-heavy, feed loads in &lt; 200 ms.



**Fan-out on write** (push): when someone posts, a worker inserts the post id into each follower's feed list in a cache — reads are a single lookup. **Fan-out on read** (pull): at read time, fetch recent posts of everyone followed and merge — no write amplification, slower reads. A celebrity with 50 million followers makes push impossible, so real systems are **hybrid**: push for normal accounts, pull and merge for celebrities.



Store only post ids in feeds; hydrate the post content from a post cache. Rank with an engagement-and-recency score (or an ML model), and enforce diversity rules.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
import heapq


def merge_demo():
    followed = {"asha": [(90, "a2"), (40, "a1")], "star": [(95, "s7"), (60, "s6")], "ravi": [(70, "r1")]}
    newest = heapq.merge(*followed.values(), reverse=True)
    print([pid for _, pid in newest])


merge_demo()

['s7', 'a2', 'r1', 's6', 'a1']


**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q13](#q13), [Q14](#q14), [Q2](#q2), [Q3](#q3), [Q20](#q20). Try one now, or carry on and do them in Part 3.

## Lesson 2: Design a chat system

*Level: Core*

**Real-time path**: clients keep a WebSocket to a chat server; a session registry (Redis) maps user → server; a message goes sender → their server → recipient's server → recipient. Offline users get a push notification and fetch messages when they return.




- **Ordering**: the server assigns a per-conversation sequence number; clients sort by it and sync "everything after seq N" on reconnect.
- **Storage**: messages in a wide-column store (Cassandra, HBase) partitioned by conversation id, clustered by seq.
- **Receipts and unread counts** come from per-user read pointers.
- **Presence** comes from heartbeats with timeouts; throttle typing indicators.
- Group chats fan out to each member; very large groups switch to pull.

**✏️ Your turn:** you now have what you need for [Q4](#q4), [Q5](#q5), [Q6](#q6), [Q7](#q7), [Q15](#q15), [Q8](#q8), [Q9](#q9), [Q16](#q16), [Q10](#q10), [Q12](#q12). Try one now, or carry on and do them in Part 3.

## Lesson 3: Design a notification service

*Level: Core*

Producers (orders, marketing, security) publish events to a queue. A notification service applies user **preferences** and **channels** (push, SMS, email), rate limits and **digests**, **quiet hours** in the user's time zone, and templates; then per-channel workers call providers (APNs, FCM, SMS gateways) with retries. Invalid device tokens are removed; every send is logged for deduplication and analytics. Critical messages (OTP) skip digests and go on a priority queue.

**✏️ Your turn:** you now have what you need for [Q17](#q17), [Q18](#q18), [Q11](#q11), [Q19](#q19). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

**▶ Run this cell before the questions.** It sets up the data and tools they share.

In [3]:
from collections import Counter, defaultdict


<a id="q1"></a>
### Q1. Push or Pull: Count the Work

🟢 Warm-up · *Fan-out on write vs read*

Each user posts `posts[u]` times a day and has `followers[u]` followers. Fan-out on write inserts every post into every follower's feed. Fan-out on read makes each feed read query every followed account. Return `(write-time inserts per day, read-time queries per day)` given `reads` feed reads per day and `follows` accounts followed per reader.

**Examples**

- `followers = {'asha': 120, 'ravi': 300, 'star': 5000000}, posts = {'asha': 3, 'ravi': 1, 'star': 2}, reads = 2000000, follows = 150` → `(10000660, 300000000)`

In [ ]:
def push_vs_pull(followers, posts, reads, follows):
    """
    Each user posts posts[u] times a day and has followers[u] followers. Fan-out
    on write inserts every post into every follower's feed. Fan-out on read
    makes each feed read query every followed account. Return (write-time
    inserts per day, read-time queries per day) given reads feed reads per day
    and follows accounts followed per reader.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(push_vs_pull, [
    (({'asha': 120, 'ravi': 300, 'star': 5000000}, {'asha': 3, 'ravi': 1, 'star': 2}, 2000000, 150), (10000660, 300000000)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop and add.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`sum` of products.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loop and add.

```python
def push_vs_pull(followers, posts, reads, follows):
    writes = 0
    for u in followers:
        writes += followers[u] * posts[u]
    return (writes, reads * follows)
```

**Cost:** O(u) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `sum` of products.

```python
def push_vs_pull(followers, posts, reads, follows):
    return (sum(followers[u] * posts[u] for u in followers), reads * follows)
```

**Cost:** O(u) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop and add. That costs O(u) time and O(1) space."

"I can do better: `sum` of products. That's O(u) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/feeds_chat/push-or-pull-count-the-work)

<a id="q2"></a>
### Q2. Rank the Feed

🟢 Warm-up · *Engagement with time decay*

Score each post `(id, likes, comments, age_hours)` as (likes + 2 × comments) / (age + 2)^1.5 and return ids from highest score.

**Examples**

- `posts = [('p1', 100, 10, 1), ('p2', 900, 50, 20), ('p3', 30, 30, 0), ('p4', 5000, 100, 72)]` → `['p3', 'p1', 'p2', 'p4']`

In [ ]:
def rank_feed(posts):
    """
    Score each post (id, likes, comments, age_hours) as (likes + 2 × comments) /
    (age + 2)^1.5 and return ids from highest score.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(rank_feed, [
    (([('p1', 100, 10, 1), ('p2', 900, 50, 20), ('p3', 30, 30, 0), ('p4', 5000, 100, 72)],), ['p3', 'p1', 'p2', 'p4']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compute scores into a list, sort it.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`sorted` with a key function.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Compute scores into a list, sort it.

```python
def rank_feed(posts):
    scored = []
    for pid, likes, comments, age in posts:
        scored.append(((likes + 2 * comments) / (age + 2) ** 1.5, pid))
    scored.sort(reverse=True)
    return [pid for _, pid in scored]
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `sorted` with a key function.

```python
def rank_feed(posts):
    return [p[0] for p in sorted(posts, key=lambda p: (p[1] + 2 * p[2]) / (p[3] + 2) ** 1.5, reverse=True)]
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compute scores into a list, sort it. That costs O(n log n) time and O(n) space."

"I can do better: `sorted` with a key function. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/feeds_chat/rank-the-feed)

<a id="q3"></a>
### Q3. Keep the Feed Diverse

🟢 Warm-up · *Cap posts per author*

From a ranked list of `(post, author)`, keep at most `cap` posts per author (in ranked order) and return the first `n` post ids.

**Examples**

- `ranked = [('p1', 'news'), ('p2', 'news'), ('p3', 'asha'), ('p4', 'news'), ('p5', 'news'), ('p6', 'ravi'), ('p7', 'asha')], cap = 2, n = 5` → `['p1', 'p2', 'p3', 'p6', 'p7']`

In [ ]:
def diverse(ranked, cap, n):
    """
    From a ranked list of (post, author), keep at most cap posts per author (in
    ranked order) and return the first n post ids.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(diverse, [
    (([('p1', 'news'), ('p2', 'news'), ('p3', 'asha'), ('p4', 'news'), ('p5', 'news'), ('p6', 'ravi'), ('p7', 'asha')], 2, 5), ['p1', 'p2', 'p3', 'p6', 'p7']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count with a dict in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Counter and early stop.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Count with a dict in a loop.

```python
def diverse(ranked, cap, n):
    counts, out = {}, []
    for pid, author in ranked:
        if counts.get(author, 0) < cap:
            out.append(pid)
            counts[author] = counts.get(author, 0) + 1
    return out[:n]
```

**Cost:** O(p) time, O(a) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Counter and early stop.

```python
def diverse(ranked, cap, n):
    counts, out = Counter(), []
    for pid, author in ranked:
        if counts[author] < cap:
            counts[author] += 1
            out.append(pid)
            if len(out) == n:
                break
    return out
```

**Cost:** O(p) time, O(a) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count with a dict in a loop. That costs O(p) time and O(a) space."

"I can do better: Counter and early stop. That's O(p) time and O(a) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/feeds_chat/keep-the-feed-diverse)

<a id="q4"></a>
### Q4. Order Chat Messages

🟢 Warm-up · *Server-assigned sequence numbers*

Client clocks can't be trusted, so the chat server gives each conversation its own increasing sequence number as messages arrive. Messages are `(conversation, text)` in arrival order; return `(conversation, seq, text)`.

**Examples**

- `messages = [('c1', 'hi'), ('c2', 'yo'), ('c1', 'how are you?'), ('c1', '👋'), ('c2', 'lunch?')]` → `[('c1', 1, 'hi'), ('c2', 1, 'yo'), ('c1', 2, 'how are you?'), ('c1', 3, '👋'), ('c2', 2, 'lunch?')]`

In [ ]:
def chat_seq(messages):
    """
    Client clocks can't be trusted, so the chat server gives each conversation
    its own increasing sequence number as messages arrive. Messages are
    (conversation, text) in arrival order; return (conversation, seq, text).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(chat_seq, [
    (([('c1', 'hi'), ('c2', 'yo'), ('c1', 'how are you?'), ('c1', '👋'), ('c2', 'lunch?')],), [('c1', 1, 'hi'), ('c2', 1, 'yo'), ('c1', 2, 'how are you?'), ('c1', 3, '👋'), ('c2', 2, 'lunch?')]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count earlier messages per conversation.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A counter per conversation.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Count earlier messages per conversation.

```python
def chat_seq(messages):
    out = []
    for i, (c, text) in enumerate(messages):
        seq = 1 + sum(1 for j in range(i) if messages[j][0] == c)
        out.append((c, seq, text))
    return out
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A counter per conversation.

```python
def chat_seq(messages):
    nxt, out = Counter(), []
    for c, text in messages:
        nxt[c] += 1
        out.append((c, nxt[c], text))
    return out
```

**Cost:** O(n) time, O(c) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count earlier messages per conversation. That costs O(n²) time and O(n) space."

"I can do better: A counter per conversation. That's O(n) time and O(c) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/feeds_chat/order-chat-messages)

<a id="q5"></a>
### Q5. Unread Counts

🟢 Warm-up · *Latest seq minus last read*

For each conversation, the latest sequence number is `latest[c]` and the user's read pointer is `read[c]` (missing = 0). Return unread per conversation (only non-zero) and the badge total.

**Examples**

- `latest = {'c1': 40, 'c2': 12, 'c3': 7, 'c4': 3}, read = {'c1': 40, 'c2': 5, 'c4': 1}` → `({'c2': 7, 'c3': 7, 'c4': 2}, 16)`

In [ ]:
def unread(latest, read):
    """
    For each conversation, the latest sequence number is latest[c] and the
    user's read pointer is read[c] (missing = 0). Return unread per conversation
    (only non-zero) and the badge total.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(unread, [
    (({'c1': 40, 'c2': 12, 'c3': 7, 'c4': 3}, {'c1': 40, 'c2': 5, 'c4': 1}), ({'c2': 7, 'c3': 7, 'c4': 2}, 16)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A dict comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A loop.

```python
def unread(latest, read):
    out = {}
    for c, s in latest.items():
        n = s - read.get(c, 0)
        if n > 0:
            out[c] = n
    total = 0
    for n in out.values():
        total += n
    return (out, total)
```

**Cost:** O(c) time, O(c) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A dict comprehension.

```python
def unread(latest, read):
    out = {c: s - read.get(c, 0) for c, s in latest.items() if s > read.get(c, 0)}
    return (out, sum(out.values()))
```

**Cost:** O(c) time, O(c) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop. That costs O(c) time and O(c) space."

"I can do better: A dict comprehension. That's O(c) time and O(c) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/feeds_chat/unread-counts)

<a id="q6"></a>
### Q6. Delivery Receipts

🟢 Warm-up · *Sent → delivered → read*

Receipt events `(message, state)` may arrive out of order. A message's state only moves forward (sent &lt; delivered &lt; read). Return each message's final state, sorted by message.

**Examples**

- `events = [('m1', 'sent'), ('m2', 'sent'), ('m1', 'read'), ('m1', 'delivered'), ('m2', 'delivered'), ('m3', 'sent'), ('m2', 'sent')]` → `[('m1', 'read'), ('m2', 'delivered'), ('m3', 'sent')]`

In [ ]:
def receipts(events):
    """
    Receipt events (message, state) may arrive out of order. A message's state
    only moves forward (sent < delivered < read). Return each message's final
    state, sorted by message.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(receipts, [
    (([('m1', 'sent'), ('m2', 'sent'), ('m1', 'read'), ('m1', 'delivered'), ('m2', 'delivered'), ('m3', 'sent'), ('m2', 'sent')],), [('m1', 'read'), ('m2', 'delivered'), ('m3', 'sent')]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compare ranks with if-statements.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`max` by rank.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Compare ranks with if-statements.

```python
def receipts(events):
    rank = {"sent": 0, "delivered": 1, "read": 2}
    state = {}
    for m, s in events:
        if m not in state or rank[s] > rank[state[m]]:
            state[m] = s
    return sorted(state.items())
```

**Cost:** O(e) time, O(m) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `max` by rank.

```python
def receipts(events):
    order = ["sent", "delivered", "read"]
    state = {}
    for m, s in events:
        state[m] = max(state.get(m, "sent"), s, key=order.index)
    return sorted(state.items())
```

**Cost:** O(e) time, O(m) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compare ranks with if-statements. That costs O(e) time and O(m) space."

"I can do better: `max` by rank. That's O(e) time and O(m) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/feeds_chat/delivery-receipts)

<a id="q7"></a>
### Q7. Who Is Online?

🟢 Warm-up · *Presence from heartbeats*

Clients send heartbeats. A user is 'online' if their last heartbeat was within 30 s of `now`, 'away' within 300 s, else 'offline' (never seen = offline). Return the status of each user in `users`.

**Examples**

- `beats = [('asha', 100), ('ravi', 50), ('asha', 990), ('meera', 800), ('ravi', 700)], now = 1000, users = ['asha', 'ravi', 'meera', 'dev']` → `['online', 'away', 'away', 'offline']`

In [ ]:
def presence(beats, now, users):
    """
    Clients send heartbeats. A user is 'online' if their last heartbeat was
    within 30 s of now, 'away' within 300 s, else 'offline' (never seen =
    offline). Return the status of each user in users.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(presence, [
    (([('asha', 100), ('ravi', 50), ('asha', 990), ('meera', 800), ('ravi', 700)], 1000, ['asha', 'ravi', 'meera', 'dev']), ['online', 'away', 'away', 'offline']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Find each user's last heartbeat by scanning.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A dict of last heartbeats.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Find each user's last heartbeat by scanning.

```python
def presence(beats, now, users):
    out = []
    for u in users:
        last = None
        for name, t in beats:
            if name == u and (last is None or t > last):
                last = t
        if last is not None and now - last <= 30:
            out.append("online")
        elif last is not None and now - last <= 300:
            out.append("away")
        else:
            out.append("offline")
    return out
```

**Cost:** O(u · h) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A dict of last heartbeats.

```python
def presence(beats, now, users):
    last = {}
    for u, t in beats:
        last[u] = max(t, last.get(u, t))
    def status(u):
        gap = now - last.get(u, -math.inf)
        return "online" if gap <= 30 else "away" if gap <= 300 else "offline"
    return [status(u) for u in users]
```

**Cost:** O(h + u) time, O(u) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Find each user's last heartbeat by scanning. That costs O(u · h) time and O(1) space."

"I can do better: A dict of last heartbeats. That's O(h + u) time and O(u) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `beats = [('a', 95)], now = 100, users = ['a', 'b']`

1. status() returns 'online'
2. status() returns 'offline'
3. presence() returns ['online', 'offline']

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/feeds_chat/who-is-online)

<a id="q8"></a>
### Q8. Group Chat Fan-Out

🟢 Warm-up · *Every member gets a copy*

Groups map to member lists. Each message `(group, sender)` is delivered to every member except the sender. Return the total deliveries and each user's inbox count (sorted).

**Examples**

- `groups = {'family': ['asha', 'ravi', 'meera'], 'team': ['asha', 'dev', 'kim', 'lee']}, messages = [('family', 'asha'), ('team', 'dev'), ('team', 'asha'), ('family', 'meera')]` → `(10, [('asha', 2), ('dev', 1), ('kim', 2), ('lee', 2), ('meera', 1), ('ravi', 2)])`

In [ ]:
def group_fanout(groups, messages):
    """
    Groups map to member lists. Each message (group, sender) is delivered to
    every member except the sender. Return the total deliveries and each user's
    inbox count (sorted).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(group_fanout, [
    (({'family': ['asha', 'ravi', 'meera'], 'team': ['asha', 'dev', 'kim', 'lee']}, [('family', 'asha'), ('team', 'dev'), ('team', 'asha'), ('family', 'meera')]), (10, [('asha', 2), ('dev', 1), ('kim', 2), ('lee', 2), ('meera', 1), ('ravi', 2)])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Nested loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Counter update per message.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Nested loops.

```python
def group_fanout(groups, messages):
    inbox = {}
    for g, sender in messages:
        for m in groups[g]:
            if m != sender:
                inbox[m] = inbox.get(m, 0) + 1
    return (sum(inbox.values()), sorted(inbox.items()))
```

**Cost:** O(m · g) time, O(u) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Counter update per message.

```python
def group_fanout(groups, messages):
    inbox = Counter(m for g, s in messages for m in groups[g] if m != s)
    return (sum(inbox.values()), sorted(inbox.items()))
```

**Cost:** O(m · g) time, O(u) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Nested loops. That costs O(m · g) time and O(u) space."

"I can do better: Counter update per message. That's O(m · g) time and O(u) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/feeds_chat/group-chat-fan-out)

<a id="q9"></a>
### Q9. How Many WebSocket Servers?

🟢 Warm-up · *Long-lived connections*

Return `(servers needed, total memory in GB rounded to 1)` for `online` concurrent users, at most `per_server` connections per server and `kb_each` KB of memory per connection.

**Examples**

- `online = 10000000, per_server = 50000, kb_each = 64` → `(200, 610.4)`
- `online = 250000, per_server = 100000, kb_each = 20` → `(3, 4.8)`

In [ ]:
def ws_servers(online, per_server, kb_each):
    """
    Return (servers needed, total memory in GB rounded to 1) for online
    concurrent users, at most per_server connections per server and kb_each KB
    of memory per connection.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(ws_servers, [
    ((10000000, 50000, 64), (200, 610.4)),
    ((250000, 100000, 20), (3, 4.8)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count servers in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`math.ceil`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Count servers in a loop.

```python
def ws_servers(online, per_server, kb_each):
    servers = 0
    while servers * per_server < online:
        servers += 1
    return (servers, round(online * kb_each / 1024 / 1024, 1))
```

**Cost:** O(s) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `math.ceil`.

```python
def ws_servers(online, per_server, kb_each):
    return (math.ceil(online / per_server), round(online * kb_each / 2 ** 20, 1))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count servers in a loop. That costs O(s) time and O(1) space."

"I can do better: `math.ceil`. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/feeds_chat/how-many-websocket-servers)

<a id="q10"></a>
### Q10. Throttle the Typing Indicator

🟢 Warm-up · *At most one event per interval*

Keystroke times (ms) would each send a 'typing…' event. Send one only if at least `interval` ms passed since the last one sent. Return the send times.

**Examples**

- `times = [0, 100, 250, 900, 1000, 1500, 3200, 3300], interval = 1000` → `[0, 1000, 3200]`

In [ ]:
def throttle(times, interval):
    """
    Keystroke times (ms) would each send a 'typing…' event. Send one only if at
    least interval ms passed since the last one sent. Return the send times.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(throttle, [
    (([0, 100, 250, 900, 1000, 1500, 3200, 3300], 1000), [0, 1000, 3200]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Track the last sent time.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same, one condition.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Track the last sent time.

```python
def throttle(times, interval):
    sent = []
    for t in times:
        if not sent or t - sent[-1] >= interval:
            sent.append(t)
    return sent
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same, one condition.

```python
def throttle(times, interval):
    sent, last = [], -math.inf
    for t in times:
        if t - last >= interval:
            sent.append(t)
            last = t
    return sent
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Track the last sent time. That costs O(n) time and O(n) space."

"I can do better: Same, one condition. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/feeds_chat/throttle-the-typing-indicator)

<a id="q11"></a>
### Q11. Choose Notification Channels

🟢 Warm-up · *User preferences*

Events are `(user, type)`. Use the user's preference for that type if set, else `defaults[type]`. Return the channels per event and the total sends per channel (sorted).

**Examples**

- `events = [('asha', 'otp'), ('asha', 'promo'), ('ravi', 'order'), ('ravi', 'promo'), ('meera', 'otp')], prefs = {'asha': {'promo': []}, 'ravi': {'order': ['email']}}, defaults = {'otp': ['sms'], 'order': ['push', 'email'], 'promo': ['push']}` → `([['sms'], [], ['email'], ['push'], ['sms']], [('email', 1), ('push', 1), ('sms', 2)])`

In [ ]:
def channels(events, prefs, defaults):
    """
    Events are (user, type). Use the user's preference for that type if set,
    else defaults[type]. Return the channels per event and the total sends per
    channel (sorted).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(channels, [
    (([('asha', 'otp'), ('asha', 'promo'), ('ravi', 'order'), ('ravi', 'promo'), ('meera', 'otp')], {'asha': {'promo': []}, 'ravi': {'order': ['email']}}, {'otp': ['sms'], 'order': ['push', 'email'], 'promo': ['push']}), ([['sms'], [], ['email'], ['push'], ['sms']], [('email', 1), ('push', 1), ('sms', 2)])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Nested ifs and a counting dict.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`dict.get` chains and Counter.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Nested ifs and a counting dict.

```python
def channels(events, prefs, defaults):
    out, totals = [], {}
    for u, t in events:
        if u in prefs and t in prefs[u]:
            ch = prefs[u][t]
        else:
            ch = defaults[t]
        out.append(ch)
        for c in ch:
            totals[c] = totals.get(c, 0) + 1
    return (out, sorted(totals.items()))
```

**Cost:** O(e · c) time, O(c) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `dict.get` chains and Counter.

```python
def channels(events, prefs, defaults):
    out = [prefs.get(u, {}).get(t, defaults[t]) for u, t in events]
    return (out, sorted(Counter(c for ch in out for c in ch).items()))
```

**Cost:** O(e · c) time, O(c) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Nested ifs and a counting dict. That costs O(e · c) time and O(c) space."

"I can do better: `dict.get` chains and Counter. That's O(e · c) time and O(c) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/feeds_chat/choose-notification-channels)

<a id="q12"></a>
### Q12. How Much Chat Storage?

🟢 Warm-up · *Size the message store*

Return `(messages per day, write QPS, TB per year)` for `dau` users sending `per_user` messages a day of `bytes_each` bytes, stored `copies` times. Round QPS and TB to 1.

**Examples**

- `dau = 500000000, per_user = 40, bytes_each = 200, copies = 3` → `(20000000000, 231481.5, 4380.0)`

In [ ]:
def chat_storage(dau, per_user, bytes_each, copies):
    """
    Return (messages per day, write QPS, TB per year) for dau users sending
    per_user messages a day of bytes_each bytes, stored copies times. Round QPS
    and TB to 1.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(chat_storage, [
    ((500000000, 40, 200, 3), (20000000000, 231481.5, 4380.0)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Step-by-step arithmetic.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same, compact.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Step-by-step arithmetic.

```python
def chat_storage(dau, per_user, bytes_each, copies):
    per_day = dau * per_user
    qps_ = per_day / (24 * 3600)
    tb = per_day * 365 * bytes_each * copies / 1_000_000_000_000
    return (per_day, round(qps_, 1), round(tb, 1))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same, compact.

```python
def chat_storage(dau, per_user, bytes_each, copies):
    n = dau * per_user
    return (n, round(n / 86400, 1), round(n * 365 * bytes_each * copies / 1e12, 1))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Step-by-step arithmetic. That costs O(1) time and O(1) space."

"I can do better: Same, compact. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/feeds_chat/how-much-chat-storage)

<a id="q13"></a>
### Q13. Hybrid Fan-Out for Celebrities

🟡 Interview standard · *Push most, pull the famous*

Accounts with more than `limit` followers are 'pull' (their posts are merged at read time); the rest are 'push'. Return the sorted pull accounts and the number of feed inserts per day for the push accounts.

**Examples**

- `followers = {'asha': 120, 'ravi': 300, 'star': 5000000, 'brand': 900000}, posts = {'asha': 3, 'ravi': 1, 'star': 2, 'brand': 10}, limit = 100000` → `(['brand', 'star'], 660)`

In [ ]:
def hybrid_fanout(followers, posts, limit):
    """
    Accounts with more than limit followers are 'pull' (their posts are merged
    at read time); the rest are 'push'. Return the sorted pull accounts and the
    number of feed inserts per day for the push accounts.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(hybrid_fanout, [
    (({'asha': 120, 'ravi': 300, 'star': 5000000, 'brand': 900000}, {'asha': 3, 'ravi': 1, 'star': 2, 'brand': 10}, 100000), (['brand', 'star'], 660)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop with if/else.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Comprehensions.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loop with if/else.

```python
def hybrid_fanout(followers, posts, limit):
    pull, inserts = [], 0
    for u, f in followers.items():
        if f > limit:
            pull.append(u)
        else:
            inserts += f * posts[u]
    return (sorted(pull), inserts)
```

**Cost:** O(u log u) time, O(u) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Comprehensions.

```python
def hybrid_fanout(followers, posts, limit):
    return (sorted(u for u, f in followers.items() if f > limit),
            sum(f * posts[u] for u, f in followers.items() if f <= limit))
```

**Cost:** O(u log u) time, O(u) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop with if/else. That costs O(u log u) time and O(u) space."

"I can do better: Comprehensions. That's O(u log u) time and O(u) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/feeds_chat/hybrid-fan-out-for-celebrities)

<a id="q14"></a>
### Q14. Merge a Pull Timeline

🟡 Interview standard · *k-way merge*

Each followed account's posts are `(timestamp, id)`, newest first. Return the `k` newest post ids across the accounts the reader follows.

**Examples**

- `posts = {'a': [(50, 'a3'), (20, 'a2'), (5, 'a1')], 'b': [(45, 'b2'), (44, 'b1')], 'c': [(60, 'c9'), (10, 'c1')], 'd': [(99, 'd1')]}, following = ['a', 'b', 'c'], k = 4` → `['c9', 'a3', 'b2', 'b1']`

In [ ]:
import heapq

from itertools import islice

def pull_timeline(posts, following, k):
    """
    Each followed account's posts are (timestamp, id), newest first. Return the
    k newest post ids across the accounts the reader follows.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(pull_timeline, [
    (({'a': [(50, 'a3'), (20, 'a2'), (5, 'a1')], 'b': [(45, 'b2'), (44, 'b1')], 'c': [(60, 'c9'), (10, 'c1')], 'd': [(99, 'd1')]}, ['a', 'b', 'c'], 4), ['c9', 'a3', 'b2', 'b1']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Gather all posts and sort.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`heapq.merge(..., reverse=True)`, take k.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Gather all posts and sort.

```python
def pull_timeline(posts, following, k):
    allp = []
    for u in following:
        allp += posts[u]
    allp.sort(reverse=True)
    return [pid for _, pid in allp[:k]]
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `heapq.merge(..., reverse=True)`, take k.

```python
def pull_timeline(posts, following, k):
    return [pid for _, pid in islice(heapq.merge(*(posts[u] for u in following), reverse=True), k)]
```

**Cost:** O(k log f) time, O(f) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Gather all posts and sort. That costs O(n log n) time and O(n) space."

"I can do better: `heapq.merge(..., reverse=True)`, take k. That's O(k log f) time and O(f) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/feeds_chat/merge-a-pull-timeline)

<a id="q15"></a>
### Q15. Route Messages to Connection Servers

🟡 Interview standard · *Session registry*

Online users hold a WebSocket to one chat server (`sessions`: user → server). For each message `(sender, recipient)`, deliver it via the recipient's server, or queue a push notification if they're offline. Return `(deliveries per server, offline recipients notified, in order)`.

**Examples**

- `sessions = {'asha': 'ws-1', 'ravi': 'ws-2', 'meera': 'ws-1'}, messages = [('asha', 'ravi'), ('ravi', 'meera'), ('asha', 'dev'), ('meera', 'asha'), ('ravi', 'dev')]` → `({'ws-1': 2, 'ws-2': 1}, ['dev', 'dev'])`

In [ ]:
def route_messages(sessions, messages):
    """
    Online users hold a WebSocket to one chat server (sessions: user → server).
    For each message (sender, recipient), deliver it via the recipient's server,
    or queue a push notification if they're offline. Return (deliveries per
    server, offline recipients notified, in order).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(route_messages, [
    (({'asha': 'ws-1', 'ravi': 'ws-2', 'meera': 'ws-1'}, [('asha', 'ravi'), ('ravi', 'meera'), ('asha', 'dev'), ('meera', 'asha'), ('ravi', 'dev')]), ({'ws-1': 2, 'ws-2': 1}, ['dev', 'dev'])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop with dict counting.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A Counter and a comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A loop with dict counting.

```python
def route_messages(sessions, messages):
    per_server, offline = {}, []
    for _, to in messages:
        if to in sessions:
            per_server[sessions[to]] = per_server.get(sessions[to], 0) + 1
        else:
            offline.append(to)
    return (dict(sorted(per_server.items())), offline)
```

**Cost:** O(m) time, O(s) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A Counter and a comprehension.

```python
def route_messages(sessions, messages):
    c = Counter(sessions[to] for _, to in messages if to in sessions)
    return (dict(sorted(c.items())), [to for _, to in messages if to not in sessions])
```

**Cost:** O(m) time, O(s) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop with dict counting. That costs O(m) time and O(s) space."

"I can do better: A Counter and a comprehension. That's O(m) time and O(s) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/feeds_chat/route-messages-to-connection-servers)

<a id="q16"></a>
### Q16. Sync After Reconnecting

🟡 Interview standard · *Catch up from the last seq*

After reconnecting, a client sends its last seen seq per conversation; the server returns missed messages in pages of `page` per request. Return `(missed per conversation, requests needed)` (conversations with nothing missed need no request).

**Examples**

- `latest = {'c1': 120, 'c2': 45, 'c3': 9}, seen = {'c1': 20, 'c2': 45, 'c3': 0}, page = 50` → `({'c1': 100, 'c2': 0, 'c3': 9}, 3)`

In [ ]:
def reconnect_sync(latest, seen, page):
    """
    After reconnecting, a client sends its last seen seq per conversation; the
    server returns missed messages in pages of page per request. Return (missed
    per conversation, requests needed) (conversations with nothing missed need
    no request).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(reconnect_sync, [
    (({'c1': 120, 'c2': 45, 'c3': 9}, {'c1': 20, 'c2': 45, 'c3': 0}, 50), ({'c1': 100, 'c2': 0, 'c3': 9}, 3)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop with page counting.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Ceiling division.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loop with page counting.

```python
def reconnect_sync(latest, seen, page):
    missed, requests = {}, 0
    for c in latest:
        m = latest[c] - seen.get(c, 0)
        missed[c] = m
        while m > 0:
            requests += 1
            m -= page
    return (missed, requests)
```

**Cost:** O(c · p) time, O(c) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Ceiling division.

```python
def reconnect_sync(latest, seen, page):
    missed = {c: s - seen.get(c, 0) for c, s in latest.items()}
    return (missed, sum(math.ceil(m / page) for m in missed.values()))
```

**Cost:** O(c) time, O(c) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop with page counting. That costs O(c · p) time and O(c) space."

"I can do better: Ceiling division. That's O(c) time and O(c) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/feeds_chat/sync-after-reconnecting)

<a id="q17"></a>
### Q17. Notification Digests

🟡 Interview standard · *Don't buzz the phone 30 times*

Per user, the first notification sends immediately and opens a window of `window` s; further notifications inside the window are collapsed into one digest sent when the window ends. Return `(user, send_time, count)` sorted by user and time.

**Examples**

- `notes = [('asha', 0), ('asha', 10), ('asha', 50), ('ravi', 5), ('asha', 70), ('asha', 200), ('ravi', 300)], window = 60` → `[('asha', 0, 1), ('asha', 60, 2), ('asha', 70, 1), ('asha', 200, 1), ('ravi', 5, 1), ('ravi', 300, 1)]`

In [ ]:
def digests(notes, window):
    """
    Per user, the first notification sends immediately and opens a window of
    window s; further notifications inside the window are collapsed into one
    digest sent when the window ends. Return (user, send_time, count) sorted by
    user and time.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(digests, [
    (([('asha', 0), ('asha', 10), ('asha', 50), ('ravi', 5), ('asha', 70), ('asha', 200), ('ravi', 300)], 60), [('asha', 0, 1), ('asha', 60, 2), ('asha', 70, 1), ('asha', 200, 1), ('ravi', 5, 1), ('ravi', 300, 1)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Group per user, then walk their times.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same with `defaultdict(list)`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Group per user, then walk their times.

```python
def digests(notes, window):
    users = sorted(set(u for u, _ in notes))
    out = []
    for u in users:
        times = sorted(t for x, t in notes if x == u)
        i = 0
        while i < len(times):
            start = times[i]
            out.append((u, start, 1))
            j = i + 1
            while j < len(times) and times[j] < start + window:
                j += 1
            if j - i - 1 > 0:
                out.append((u, start + window, j - i - 1))
            i = j
    return out
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same with `defaultdict(list)`.

```python
def digests(notes, window):
    per = defaultdict(list)
    for u, t in notes:
        per[u].append(t)
    out = []
    for u in sorted(per):
        times, i = sorted(per[u]), 0
        while i < len(times):
            start, j = times[i], i + 1
            while j < len(times) and times[j] < start + window:
                j += 1
            out.append((u, start, 1))
            if j > i + 1:
                out.append((u, start + window, j - i - 1))
            i = j
    return out
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Group per user, then walk their times. That costs O(n log n) time and O(n) space."

"I can do better: Same with `defaultdict(list)`. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/feeds_chat/notification-digests)

<a id="q18"></a>
### Q18. Respect Quiet Hours

🟡 Interview standard · *Time zones matter*

Notifications are `(user, utc_hour)`; `tz` gives each user's UTC offset in whole hours. Quiet hours are 22:00–07:00 local. Return the UTC hour each notification is delivered (now, or 07:00 local the same/next morning), modulo 24.

**Examples**

- `notes = [('asha', 18), ('ravi', 3), ('meera', 12), ('asha', 1)], tz = {'asha': 5, 'ravi': -5, 'meera': 9}` → `[2, 12, 12, 2]`

In [ ]:
def quiet_hours(notes, tz):
    """
    Notifications are (user, utc_hour); tz gives each user's UTC offset in whole
    hours. Quiet hours are 22:00–07:00 local. Return the UTC hour each
    notification is delivered (now, or 07:00 local the same/next morning),
    modulo 24.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(quiet_hours, [
    (([('asha', 18), ('ravi', 3), ('meera', 12), ('asha', 1)], {'asha': 5, 'ravi': -5, 'meera': 9}), [2, 12, 12, 2]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Step forward hour by hour until not quiet.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Compute local hour and shift directly.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Step forward hour by hour until not quiet.

```python
def quiet_hours(notes, tz):
    out = []
    for u, h in notes:
        local = (h + tz[u]) % 24
        while local >= 22 or local < 7:
            local = (local + 1) % 24
        out.append((local - tz[u]) % 24)
    return out
```

**Cost:** O(n · 24) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Compute local hour and shift directly.

```python
def quiet_hours(notes, tz):
    def deliver(u, h):
        local = (h + tz[u]) % 24
        return h if 7 <= local < 22 else (7 - tz[u]) % 24
    return [deliver(u, h) for u, h in notes]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Step forward hour by hour until not quiet. That costs O(n · 24) time and O(n) space."

"I can do better: Compute local hour and shift directly. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/feeds_chat/respect-quiet-hours)

<a id="q19"></a>
### Q19. Handle Push-Token Responses

🟡 Interview standard · *Clean up dead devices*

Each device token gets responses for successive attempts: 'ok' (delivered), 'invalid' (uninstalled — delete the token) or 'unavailable' (retry). Try at most `max_attempts`. Return sorted lists `(delivered, removed, gave up)`.

**Examples**

- `responses = {'t1': ['ok'], 't2': ['unavailable', 'ok'], 't3': ['invalid'], 't4': ['unavailable', 'unavailable', 'unavailable', 'ok'], 't5': ['unavailable', 'invalid']}, max_attempts = 3` → `(['t1', 't2'], ['t3', 't5'], ['t4'])`

In [ ]:
def push_tokens(responses, max_attempts):
    """
    Each device token gets responses for successive attempts: 'ok' (delivered),
    'invalid' (uninstalled — delete the token) or 'unavailable' (retry). Try at
    most max_attempts. Return sorted lists (delivered, removed, gave up).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(push_tokens, [
    (({'t1': ['ok'], 't2': ['unavailable', 'ok'], 't3': ['invalid'], 't4': ['unavailable', 'unavailable', 'unavailable', 'ok'], 't5': ['unavailable', 'invalid']}, 3), (['t1', 't2'], ['t3', 't5'], ['t4'])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over attempts per token.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Find the first final response in the allowed attempts.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loop over attempts per token.

```python
def push_tokens(responses, max_attempts):
    delivered, removed, gave_up = [], [], []
    for tok, rs in responses.items():
        result = None
        for i in range(min(max_attempts, len(rs))):
            if rs[i] == "ok":
                result = "ok"
                break
            if rs[i] == "invalid":
                result = "invalid"
                break
        if result == "ok":
            delivered.append(tok)
        elif result == "invalid":
            removed.append(tok)
        else:
            gave_up.append(tok)
    return (sorted(delivered), sorted(removed), sorted(gave_up))
```

**Cost:** O(t · a) time, O(t) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Find the first final response in the allowed attempts.

```python
def push_tokens(responses, max_attempts):
    final = {t: next((r for r in rs[:max_attempts] if r != "unavailable"), "gave up") for t, rs in responses.items()}
    return tuple(sorted(t for t, f in final.items() if f == k) for k in ("ok", "invalid", "gave up"))
```

**Cost:** O(t · a) time, O(t) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over attempts per token. That costs O(t · a) time and O(t) space."

"I can do better: Find the first final response in the allowed attempts. That's O(t · a) time and O(t) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `responses = {'a': ['ok'], 'b': ['invalid']}, max_attempts = 2`

1. GeneratorExit raised: 
2. GeneratorExit raised: 
3. push\_tokens() returns (['a'], ['b'], [])

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/feeds_chat/handle-push-token-responses)

<a id="q20"></a>
### Q20. Sharded Like Counters

🟡 Interview standard · *Spread a hot counter*

One row receiving thousands of likes per second becomes a lock hotspot. Spread increments over `n` sub-counters by `crc32(event_id) % n`; a read sums them. Return the sub-counter values and the total.

**Examples**

- `events = ['like-0', 'like-1', 'like-2', 'like-3', 'like-4', 'like-5', 'like-6', 'like-7', 'like-8', 'like-9', 'like-10', 'like-11', 'like-12', 'like-13', 'like-14', 'like-15', 'like-16', 'like-17', 'like-18', 'like-19', 'like-20', 'like-21', 'like-22', 'like-23', 'like-24', 'like-25', 'like-26', 'like-27', 'like-28', 'like-29', 'like-30', 'like-31', 'like-32', 'like-33', 'like-34', 'like-35', 'like-36', 'like-37', 'like-38', 'like-39', 'like-40', 'like-41', 'like-42', 'like-43', 'like-44', 'like-45', 'like-46', 'like-47', 'like-48', 'like-49', 'like-50', 'like-51', 'like-52', 'like-53', 'like-54', 'like-55', 'like-56', 'like-57', 'like-58', 'like-59', 'like-60', 'like-61', 'like-62', 'like-63', 'like-64', 'like-65', 'like-66', 'like-67', 'like-68', 'like-69', 'like-70', 'like-71', 'like-72', 'like-73', 'like-74', 'like-75', 'like-76', 'like-77', 'like-78', 'like-79', 'like-80', 'like-81', 'like-82', 'like-83', 'like-84', 'like-85', 'like-86', 'like-87', 'like-88', 'like-89', 'like-90', 'like-91', 'like-92', 'like-93', 'like-94', 'like-95', 'like-96', 'like-97', 'like-98', 'like-99', 'like-100', 'like-101', 'like-102', 'like-103', 'like-104', 'like-105', 'like-106', 'like-107', 'like-108', 'like-109', 'like-110', 'like-111', 'like-112', 'like-113', 'like-114', 'like-115', 'like-116', 'like-117', 'like-118', 'like-119', 'like-120', 'like-121', 'like-122', 'like-123', 'like-124', 'like-125', 'like-126', 'like-127', 'like-128', 'like-129', 'like-130', 'like-131', 'like-132', 'like-133', 'like-134', 'like-135', 'like-136', 'like-137', 'like-138', 'like-139', 'like-140', 'like-141', 'like-142', 'like-143', 'like-144', 'like-145', 'like-146', 'like-147', 'like-148', 'like-149', 'like-150', 'like-151', 'like-152', 'like-153', 'like-154', 'like-155', 'like-156', 'like-157', 'like-158', 'like-159', 'like-160', 'like-161', 'like-162', 'like-163', 'like-164', 'like-165', 'like-166', 'like-167', 'like-168', 'like-169', 'like-170', 'like-171', 'like-172', 'like-173', 'like-174', 'like-175', 'like-176', 'like-177', 'like-178', 'like-179', 'like-180', 'like-181', 'like-182', 'like-183', 'like-184', 'like-185', 'like-186', 'like-187', 'like-188', 'like-189', 'like-190', 'like-191', 'like-192', 'like-193', 'like-194', 'like-195', 'like-196', 'like-197', 'like-198', 'like-199', 'like-200', 'like-201', 'like-202', 'like-203', 'like-204', 'like-205', 'like-206', 'like-207', 'like-208', 'like-209', 'like-210', 'like-211', 'like-212', 'like-213', 'like-214', 'like-215', 'like-216', 'like-217', 'like-218', 'like-219', 'like-220', 'like-221', 'like-222', 'like-223', 'like-224', 'like-225', 'like-226', 'like-227', 'like-228', 'like-229', 'like-230', 'like-231', 'like-232', 'like-233', 'like-234', 'like-235', 'like-236', 'like-237', 'like-238', 'like-239', 'like-240', 'like-241', 'like-242', 'like-243', 'like-244', 'like-245', 'like-246', 'like-247', 'like-248', 'like-249', 'like-250', 'like-251', 'like-252', 'like-253', 'like-254', 'like-255', 'like-256', 'like-257', 'like-258', 'like-259', 'like-260', 'like-261', 'like-262', 'like-263', 'like-264', 'like-265', 'like-266', 'like-267', 'like-268', 'like-269', 'like-270', 'like-271', 'like-272', 'like-273', 'like-274', 'like-275', 'like-276', 'like-277', 'like-278', 'like-279', 'like-280', 'like-281', 'like-282', 'like-283', 'like-284', 'like-285', 'like-286', 'like-287', 'like-288', 'like-289', 'like-290', 'like-291', 'like-292', 'like-293', 'like-294', 'like-295', 'like-296', 'like-297', 'like-298', 'like-299', 'like-300', 'like-301', 'like-302', 'like-303', 'like-304', 'like-305', 'like-306', 'like-307', 'like-308', 'like-309', 'like-310', 'like-311', 'like-312', 'like-313', 'like-314', 'like-315', 'like-316', 'like-317', 'like-318', 'like-319', 'like-320', 'like-321', 'like-322', 'like-323', 'like-324', 'like-325', 'like-326', 'like-327', 'like-328', 'like-329', 'like-330', 'like-331', 'like-332', 'like-333', 'like-334', 'like-335', 'like-336', 'like-337', 'like-338', 'like-339', 'like-340', 'like-341', 'like-342', 'like-343', 'like-344', 'like-345', 'like-346', 'like-347', 'like-348', 'like-349', 'like-350', 'like-351', 'like-352', 'like-353', 'like-354', 'like-355', 'like-356', 'like-357', 'like-358', 'like-359', 'like-360', 'like-361', 'like-362', 'like-363', 'like-364', 'like-365', 'like-366', 'like-367', 'like-368', 'like-369', 'like-370', 'like-371', 'like-372', 'like-373', 'like-374', 'like-375', 'like-376', 'like-377', 'like-378', 'like-379', 'like-380', 'like-381', 'like-382', 'like-383', 'like-384', 'like-385', 'like-386', 'like-387', 'like-388', 'like-389', 'like-390', 'like-391', 'like-392', 'like-393', 'like-394', 'like-395', 'like-396', 'like-397', 'like-398', 'like-399', 'like-400', 'like-401', 'like-402', 'like-403', 'like-404', 'like-405', 'like-406', 'like-407', 'like-408', 'like-409', 'like-410', 'like-411', 'like-412', 'like-413', 'like-414', 'like-415', 'like-416', 'like-417', 'like-418', 'like-419', 'like-420', 'like-421', 'like-422', 'like-423', 'like-424', 'like-425', 'like-426', 'like-427', 'like-428', 'like-429', 'like-430', 'like-431', 'like-432', 'like-433', 'like-434', 'like-435', 'like-436', 'like-437', 'like-438', 'like-439', 'like-440', 'like-441', 'like-442', 'like-443', 'like-444', 'like-445', 'like-446', 'like-447', 'like-448', 'like-449', 'like-450', 'like-451', 'like-452', 'like-453', 'like-454', 'like-455', 'like-456', 'like-457', 'like-458', 'like-459', 'like-460', 'like-461', 'like-462', 'like-463', 'like-464', 'like-465', 'like-466', 'like-467', 'like-468', 'like-469', 'like-470', 'like-471', 'like-472', 'like-473', 'like-474', 'like-475', 'like-476', 'like-477', 'like-478', 'like-479', 'like-480', 'like-481', 'like-482', 'like-483', 'like-484', 'like-485', 'like-486', 'like-487', 'like-488', 'like-489', 'like-490', 'like-491', 'like-492', 'like-493', 'like-494', 'like-495', 'like-496', 'like-497', 'like-498', 'like-499'], n = 3` → `([184, 162, 154], 500)`

In [ ]:
import zlib

def sharded_counter(events, n):
    """
    One row receiving thousands of likes per second becomes a lock hotspot.
    Spread increments over n sub-counters by crc32(event_id) % n; a read sums
    them. Return the sub-counter values and the total.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(sharded_counter, [
    ((['like-0', 'like-1', 'like-2', 'like-3', 'like-4', 'like-5', 'like-6', 'like-7', 'like-8', 'like-9', 'like-10', 'like-11', 'like-12', 'like-13', 'like-14', 'like-15', 'like-16', 'like-17', 'like-18', 'like-19', 'like-20', 'like-21', 'like-22', 'like-23', 'like-24', 'like-25', 'like-26', 'like-27', 'like-28', 'like-29', 'like-30', 'like-31', 'like-32', 'like-33', 'like-34', 'like-35', 'like-36', 'like-37', 'like-38', 'like-39', 'like-40', 'like-41', 'like-42', 'like-43', 'like-44', 'like-45', 'like-46', 'like-47', 'like-48', 'like-49', 'like-50', 'like-51', 'like-52', 'like-53', 'like-54', 'like-55', 'like-56', 'like-57', 'like-58', 'like-59', 'like-60', 'like-61', 'like-62', 'like-63', 'like-64', 'like-65', 'like-66', 'like-67', 'like-68', 'like-69', 'like-70', 'like-71', 'like-72', 'like-73', 'like-74', 'like-75', 'like-76', 'like-77', 'like-78', 'like-79', 'like-80', 'like-81', 'like-82', 'like-83', 'like-84', 'like-85', 'like-86', 'like-87', 'like-88', 'like-89', 'like-90', 'like-91', 'like-92', 'like-93', 'like-94', 'like-95', 'like-96', 'like-97', 'like-98', 'like-99', 'like-100', 'like-101', 'like-102', 'like-103', 'like-104', 'like-105', 'like-106', 'like-107', 'like-108', 'like-109', 'like-110', 'like-111', 'like-112', 'like-113', 'like-114', 'like-115', 'like-116', 'like-117', 'like-118', 'like-119', 'like-120', 'like-121', 'like-122', 'like-123', 'like-124', 'like-125', 'like-126', 'like-127', 'like-128', 'like-129', 'like-130', 'like-131', 'like-132', 'like-133', 'like-134', 'like-135', 'like-136', 'like-137', 'like-138', 'like-139', 'like-140', 'like-141', 'like-142', 'like-143', 'like-144', 'like-145', 'like-146', 'like-147', 'like-148', 'like-149', 'like-150', 'like-151', 'like-152', 'like-153', 'like-154', 'like-155', 'like-156', 'like-157', 'like-158', 'like-159', 'like-160', 'like-161', 'like-162', 'like-163', 'like-164', 'like-165', 'like-166', 'like-167', 'like-168', 'like-169', 'like-170', 'like-171', 'like-172', 'like-173', 'like-174', 'like-175', 'like-176', 'like-177', 'like-178', 'like-179', 'like-180', 'like-181', 'like-182', 'like-183', 'like-184', 'like-185', 'like-186', 'like-187', 'like-188', 'like-189', 'like-190', 'like-191', 'like-192', 'like-193', 'like-194', 'like-195', 'like-196', 'like-197', 'like-198', 'like-199', 'like-200', 'like-201', 'like-202', 'like-203', 'like-204', 'like-205', 'like-206', 'like-207', 'like-208', 'like-209', 'like-210', 'like-211', 'like-212', 'like-213', 'like-214', 'like-215', 'like-216', 'like-217', 'like-218', 'like-219', 'like-220', 'like-221', 'like-222', 'like-223', 'like-224', 'like-225', 'like-226', 'like-227', 'like-228', 'like-229', 'like-230', 'like-231', 'like-232', 'like-233', 'like-234', 'like-235', 'like-236', 'like-237', 'like-238', 'like-239', 'like-240', 'like-241', 'like-242', 'like-243', 'like-244', 'like-245', 'like-246', 'like-247', 'like-248', 'like-249', 'like-250', 'like-251', 'like-252', 'like-253', 'like-254', 'like-255', 'like-256', 'like-257', 'like-258', 'like-259', 'like-260', 'like-261', 'like-262', 'like-263', 'like-264', 'like-265', 'like-266', 'like-267', 'like-268', 'like-269', 'like-270', 'like-271', 'like-272', 'like-273', 'like-274', 'like-275', 'like-276', 'like-277', 'like-278', 'like-279', 'like-280', 'like-281', 'like-282', 'like-283', 'like-284', 'like-285', 'like-286', 'like-287', 'like-288', 'like-289', 'like-290', 'like-291', 'like-292', 'like-293', 'like-294', 'like-295', 'like-296', 'like-297', 'like-298', 'like-299', 'like-300', 'like-301', 'like-302', 'like-303', 'like-304', 'like-305', 'like-306', 'like-307', 'like-308', 'like-309', 'like-310', 'like-311', 'like-312', 'like-313', 'like-314', 'like-315', 'like-316', 'like-317', 'like-318', 'like-319', 'like-320', 'like-321', 'like-322', 'like-323', 'like-324', 'like-325', 'like-326', 'like-327', 'like-328', 'like-329', 'like-330', 'like-331', 'like-332', 'like-333', 'like-334', 'like-335', 'like-336', 'like-337', 'like-338', 'like-339', 'like-340', 'like-341', 'like-342', 'like-343', 'like-344', 'like-345', 'like-346', 'like-347', 'like-348', 'like-349', 'like-350', 'like-351', 'like-352', 'like-353', 'like-354', 'like-355', 'like-356', 'like-357', 'like-358', 'like-359', 'like-360', 'like-361', 'like-362', 'like-363', 'like-364', 'like-365', 'like-366', 'like-367', 'like-368', 'like-369', 'like-370', 'like-371', 'like-372', 'like-373', 'like-374', 'like-375', 'like-376', 'like-377', 'like-378', 'like-379', 'like-380', 'like-381', 'like-382', 'like-383', 'like-384', 'like-385', 'like-386', 'like-387', 'like-388', 'like-389', 'like-390', 'like-391', 'like-392', 'like-393', 'like-394', 'like-395', 'like-396', 'like-397', 'like-398', 'like-399', 'like-400', 'like-401', 'like-402', 'like-403', 'like-404', 'like-405', 'like-406', 'like-407', 'like-408', 'like-409', 'like-410', 'like-411', 'like-412', 'like-413', 'like-414', 'like-415', 'like-416', 'like-417', 'like-418', 'like-419', 'like-420', 'like-421', 'like-422', 'like-423', 'like-424', 'like-425', 'like-426', 'like-427', 'like-428', 'like-429', 'like-430', 'like-431', 'like-432', 'like-433', 'like-434', 'like-435', 'like-436', 'like-437', 'like-438', 'like-439', 'like-440', 'like-441', 'like-442', 'like-443', 'like-444', 'like-445', 'like-446', 'like-447', 'like-448', 'like-449', 'like-450', 'like-451', 'like-452', 'like-453', 'like-454', 'like-455', 'like-456', 'like-457', 'like-458', 'like-459', 'like-460', 'like-461', 'like-462', 'like-463', 'like-464', 'like-465', 'like-466', 'like-467', 'like-468', 'like-469', 'like-470', 'like-471', 'like-472', 'like-473', 'like-474', 'like-475', 'like-476', 'like-477', 'like-478', 'like-479', 'like-480', 'like-481', 'like-482', 'like-483', 'like-484', 'like-485', 'like-486', 'like-487', 'like-488', 'like-489', 'like-490', 'like-491', 'like-492', 'like-493', 'like-494', 'like-495', 'like-496', 'like-497', 'like-498', 'like-499'], 3), ([184, 162, 154], 500)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A list of counters.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Counter.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A list of counters.

```python
def sharded_counter(events, n):
    shards = [0] * n
    for e in events:
        shards[zlib.crc32(e.encode()) % n] += 1
    return (shards, sum(shards))
```

**Cost:** O(e) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Counter.

```python
def sharded_counter(events, n):
    c = Counter(zlib.crc32(e.encode()) % n for e in events)
    shards = [c[i] for i in range(n)]
    return (shards, sum(shards))
```

**Cost:** O(e) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A list of counters. That costs O(e) time and O(n) space."

"I can do better: Counter. That's O(e) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/feeds_chat/sharded-like-counters)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. How do you handle the celebrity problem?</b></summary>

Don't fan out their posts. Push posts from normal accounts into feeds; at read time merge in recent posts from the few celebrities the user follows, cached aggressively.

</details>

<details>
<summary><b>2. How do you keep chat messages in order?</b></summary>

A per-conversation sequence number assigned by the server (or the partition leader for that conversation). Clients order by it and detect gaps to re-fetch.

</details>

<details>
<summary><b>3. How does a message reach a user on another server?</b></summary>

Look up the recipient's connection server in the session registry and forward the message (directly or through pub/sub); if they're offline, store it and send a push notification.

</details>

<details>
<summary><b>4. Why WebSockets for chat?</b></summary>

They keep one two-way connection open, so the server can push messages instantly without polling. Use HTTP for history, uploads and other request/response work.

</details>

<details>
<summary><b>5. How would you avoid spamming users with notifications?</b></summary>

Per-user rate limits, digests, deduplication, preference and quiet-hours checks, and priority levels so only critical messages bypass them.

</details>

<details>
<summary><b>6. Where do you store the feed?</b></summary>

Precomputed feeds of post ids in a cache (Redis lists, trimmed to a few hundred), backed by a database for rebuilding; post bodies in a separate post store and cache.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Design a news feed
- [ ] I can explain: Design a chat system
- [ ] I can explain: Design a notification service

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Sync After Reconnecting
- [ ] [Q17](#q17) Notification Digests
- [ ] [Q18](#q18) Respect Quiet Hours
- [ ] [Q19](#q19) Handle Push-Token Responses
- [ ] [Q20](#q20) Sharded Like Counters

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.